# Load Data

In [1]:
import numpy as np

loaded = np.load("ecg_dataset.npz")
X = loaded["x"]
y = loaded["y"]

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


# Model Creation

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras import Input, Model

def build_residual_cnn_feature_extractor(input_shape):
    inputs = Input(shape=input_shape)
    x = layers.Conv1D(64, 3, padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    shortcut = x

    x = layers.Conv1D(64, 3, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.add([x, shortcut])  # Residual connection
    x = layers.ReLU()(x)

    x = layers.GlobalAveragePooling1D()(x)
    features = layers.Dense(128, activation='relu')(x)  # Output as feature extractor

    feature_model = Model(inputs, features, name="ResidualCNN_FeatureExtractor")
    return feature_model

# Create classification head
def build_classifier_from_features():
    inputs = Input(shape=(128,))
    outputs = layers.Dense(1, activation='sigmoid')(inputs)
    model = Model(inputs, outputs, name="ClassificationHead")
    return model

In [ ]:
feature_model = build_residual_cnn_feature_extractor((5000, 12))
classifier = build_classifier_from_features()
full_model = models.Sequential([feature_model, classifier])

full_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
        monitor='val_loss',       # Track validation loss
        patience=3,               # Stop after 3 epochs with no improvement
        restore_best_weights=True
    )
    

In [ ]:
full_model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.1, callbacks=[early_stop])